In [1]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/ACA_MOp_VISp")

library(dplyr)

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/enrichment_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/gene_mapping_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/top_corr_module_fxns.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: future


Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths



Attaching package: ‘data.table’


The following objects are masked from ‘package:reshape2’:

    dcast, melt


The following objects are masked from ‘package:dplyr’:

    between, first, last




Here I perform enrichment analysis to find modules enriched for cell type markers. 

These modules will later be used to correlate with exon PSI to define cell type-specific exons.

In [2]:
mod_def <- "PosBC"
strict <- FALSE
lfc_threshold <- 3
pval_threshold <- 0.05

## Prep DE genes

### Pseudobulk DE genes

In [3]:
pairwise_res_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/ACA_MOp_VISp/data/DE_genes/yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_dream.RDS")

ctypes <- unique(sapply(strsplit(names(pairwise_res_list), "_vs_"), "[", 1))
n_signif_wrt <- length(ctypes)-2
if (strict) {
    n_signif_wrt <- NULL
}

marker_genes_list <- prep_pairwise_DE_genes(pairwise_res_list, pval_threshold=pval_threshold, lfc_threshold=lfc_threshold, strict=strict, n_signif_wrt=n_signif_wrt)
marker_genes_list <- marker_genes_list[lengths(marker_genes_list) > 2]

set_source <- paste0("yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strict", strict, "_logFC", lfc_threshold, "_nSignif", n_signif_wrt) 

### Claude marker genes

In [3]:
set_source <- "Claude_marker_genes"

marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_mouse.RDS")

names(marker_genes_list) <- sapply(names(marker_genes_list), function(x) gsub(" ", "_", gsub("/", "_", x, fixed=TRUE)))
names(marker_genes_list)

[1] "All_Neuronal"      "All_Glutamatergic" "All_GABAergic"    
 [4] "CGE_Class"         "MGE_Class"         "L2_3_IT"          
 [7] "L4_IT"             "L5_IT"             "L5_ET"            
[10] "L5_6_NP"           "L6_CT"             "L6_IT"            
[13] "L6_IT_Car3"        "L6b"               "Lamp5"            
[16] "Lamp5_Lhx6"        "Sncg"              "Vip"              
[19] "Pax6"              "Pvalb"             "Chandelier"       
[22] "Sst"               "Sst_Chodl"         "Astro"            
[25] "Oligo"             "OPC"               "Micro_PVM"        
[28] "Endo"              "VLMC"              "Peri"

## Get enrichments

### 20pcntCells_25SD_200samples_mergeParam0.97_subsetCutoff851.92_Modules

In [33]:
network_dir <- "yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_20pcntCells_25SD_200samples_mergeParam0.97_subsetCutoff851.92_Modules"

In [35]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [37]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)
    
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
Astro,0.000000e+00,0.000000e+00,blue,Bicor-None_signum0.257_minSize10_merge_ME_0.97_20250
Micro-PVM,0.000000e+00,0.000000e+00,brown,Bicor-None_signum0.257_minSize10_merge_ME_0.97_20250
Endo,0.000000e+00,0.000000e+00,turquoise,Bicor-None_signum0.257_minSize10_merge_ME_0.97_20250
SMC-Peri,4.145293e-231,4.402441e-228,brown,Bicor-None_signum0.971_minSize10_merge_ME_0.97_20250
Oligo,5.613860e-208,5.585554e-205,green,Bicor-None_signum0.92_minSize6_merge_ME_0.97_20250
VLMC,1.115298e-168,8.858747e-166,red,Bicor-None_signum0.92_minSize6_merge_ME_0.97_20250
Sst,1.133229e-146,8.434172e-144,black,Bicor-None_signum0.971_minSize3_merge_ME_0.97_20250
Sst_Chodl,1.061343e-77,5.364665e-75,cyan,Bicor-None_signum0.741_minSize3_merge_ME_0.97_20250
Car3,1.193794e-67,5.613859e-65,palevioletred3,Bicor-None_signum0.343_minSize10_merge_ME_0.97_20250


In [38]:
write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments.csv"), row.names=FALSE, quote=FALSE)

### 10pcntCells_5SD_200samples_mergeParam0.95_subsetCutoff763.45_Modules

In [15]:
network_dir <- "yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_5SD_200samples_mergeParam0.95_subsetCutoff763.45_Modules"

In [26]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [27]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
Endo,0.000000e+00,0.000000e+00,turquoise,Bicor-None_signum0.135_minSize10_merge_ME_0.95_18409
Micro-PVM,0.000000e+00,0.000000e+00,turquoise,Bicor-None_signum0.742_minSize10_merge_ME_0.95_18409
SMC-Peri,1.457365e-230,3.087047e-227,brown,Bicor-None_signum0.742_minSize10_merge_ME_0.95_18409
VLMC,5.442796e-206,1.086195e-202,yellow,Bicor-None_signum0.742_minSize8_merge_ME_0.95_18409
Oligo,1.608078e-197,2.934100e-194,green,Bicor-None_signum0.423_minSize10_merge_ME_0.95_18409
Astro,6.738938e-189,1.195431e-185,lightcyan,Bicor-None_signum0.742_minSize4_merge_ME_0.95_18409
Sst_Chodl,8.636439e-64,2.626341e-61,orangered4,Bicor-None_signum0.423_minSize6_merge_ME_0.95_18409
Sst,1.084343e-61,3.091393e-59,floralwhite,Bicor-None_signum0.742_minSize3_merge_ME_0.95_18409
Car3,1.012593e-60,2.717026e-58,maroon,Bicor-None_signum0.175_minSize8_merge_ME_0.95_18409


In [28]:
write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments.csv"), row.names=FALSE, quote=FALSE)

### 10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules

##### Fisher method

In [4]:
network_dir <- "yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules"

In [5]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [6]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
Oligo,1.757007e-29,1.123940e-24,red,Bicor-None_signum0.438_minSize4_merge_ME_0.95_18409
Micro_PVM,2.246532e-29,1.123940e-24,turquoise,Bicor-None_signum0.438_minSize4_merge_ME_0.95_18409
Astro,2.922182e-24,9.867549e-21,brown,Bicor-None_signum0.724_minSize8_merge_ME_0.95_18409
Endo,3.637903e-22,9.254455e-19,blue,Bicor-None_signum0.438_minSize4_merge_ME_0.95_18409
VLMC,3.230714e-18,4.998947e-15,green,Bicor-None_signum0.438_minSize4_merge_ME_0.95_18409
All_GABAergic,8.212922e-16,1.022962e-12,orangered1,Bicor-None_signum0.438_minSize8_merge_ME_0.95_18409
OPC,1.770810e-14,1.983440e-11,paleturquoise,Bicor-None_signum0.9_minSize3_merge_ME_0.95_18409
Peri,3.134024e-13,3.384069e-10,yellow,Bicor-None_signum0.35_minSize10_merge_ME_0.95_18409
Pvalb,6.404358e-13,6.769226e-10,chocolate3,Bicor-None_signum0.438_minSize6_merge_ME_0.95_18409


In [8]:
write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments.csv"), row.names=FALSE, quote=FALSE)

In [7]:
set_source

[1] "Claude_marker_genes"

##### fgsea

In [ ]:
network_dir <- "yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules"

enrichments_df <- get_module_enrichments_fgsea_par(network_dir, marker_genes_list, n_workers=20)

In [ ]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Padj, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Padj, with_ties=FALSE) %>%
    arrange(Padj)

In [ ]:
write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_fgsea.csv"), row.names=FALSE)